# 55 - Late Fusion (Transfer Learning) conf60 4-Class

Late Fusion: weighted average CNN + FCNN
**Dataset:** Front-only conf60 4-Class
**Skenario:** B1 (Baseline), B2 (Class Weights), B3 (Weights + Augmentasi)

In [1]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, accuracy_score

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from training.models import EmotionCNNTransfer, EmotionFCNN
from training.utils import EmotionMultimodalDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DATASET_DIR = PROJECT_ROOT / "data" / "dataset_frontonly_conf60_4class"
OUTPUT_DIR = PROJECT_ROOT / "models" / "frontonly_conf60" / "4class_tl"
CNN_DIR = OUTPUT_DIR
FCNN_DIR = PROJECT_ROOT / "models" / "frontonly_conf60" / "4class"  # CNN and FCNN saved here
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 32
NUM_CLASSES = 4
EMOTIONS = ["neutral", "happy", "sad", "negative"]
CNN_PREFIX = "cnn_tl"
PREFIX = "late_fusion_tl"

# Load test multimodal
test_ds = EmotionMultimodalDataset(
    DATASET_DIR / "X_test_images.npy",
    DATASET_DIR / "X_test_landmarks.npy",
    DATASET_DIR / "y_test.npy")
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Output: {OUTPUT_DIR}")
print(f"Test samples: {len(test_ds)}")

Device: cuda
GPU: Tesla T4


Output: /home/bs000716/MOTHER-TANK/TRAIN/models/frontonly_conf60/4class_tl
Test samples: 929


## Late Fusion B1, B2, B3

In [2]:
def evaluate_late_fusion(scenario):
    cnn_path = CNN_DIR / f"{CNN_PREFIX}_{scenario}.pth"
    fcnn_path = FCNN_DIR / f"fcnn_{scenario}.pth"

    if not cnn_path.exists():
        print(f"SKIP {scenario}: {cnn_path.name} not found")
        return None
    if not fcnn_path.exists():
        print(f"SKIP {scenario}: {fcnn_path.name} not found")
        return None

    cnn_model = EmotionCNNTransfer(num_classes=NUM_CLASSES).to(device)
    fcnn_model = EmotionFCNN(num_classes=NUM_CLASSES).to(device)
    cnn_model.load_state_dict(torch.load(cnn_path, map_location=device, weights_only=True))
    fcnn_model.load_state_dict(torch.load(fcnn_path, map_location=device, weights_only=True))
    cnn_model.eval(); fcnn_model.eval()

    cnn_probs_all, fcnn_probs_all, labels_all = [], [], []
    with torch.no_grad():
        for images, landmarks, labels in test_loader:
            cnn_probs_all.append(torch.softmax(cnn_model(images.to(device)), dim=1).cpu().numpy())
            fcnn_probs_all.append(torch.softmax(fcnn_model(landmarks.to(device)), dim=1).cpu().numpy())
            labels_all.append(labels.numpy())

    cnn_probs = np.concatenate(cnn_probs_all)
    fcnn_probs = np.concatenate(fcnn_probs_all)
    lbls = np.concatenate(labels_all)

    best_f1, best_w = 0, 0.5
    for w in np.arange(0.0, 1.05, 0.05):
        preds = (w * cnn_probs + (1-w) * fcnn_probs).argmax(axis=1)
        f1 = f1_score(lbls, preds, average="macro", zero_division=0)
        if f1 > best_f1: best_f1 = f1; best_w = w; best_preds = preds

    acc = accuracy_score(lbls, best_preds)
    wf1 = f1_score(lbls, best_preds, average="weighted", zero_division=0)
    print(f"  {scenario.upper()}: w={best_w:.2f} Acc={acc:.4f} Macro-F1={best_f1:.4f} W-F1={wf1:.4f}")
    return {"accuracy": float(acc), "macro_f1": float(best_f1), "weighted_f1": float(wf1), "best_cnn_weight": float(best_w)}

# Run B1, B2, B3
all_results = {}
print("\nRunning Late Fusion B1, B2, B3...")
for sc_key, sc_name in [("b1", "B1 Baseline"), ("b2", "B2 Class Weights"), ("b3", "B3 Weights+Aug")]:
    r = evaluate_late_fusion(sc_key)
    if r: all_results[sc_name] = r

# Save
with open(OUTPUT_DIR / f"{PREFIX}_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nSaved: {OUTPUT_DIR / (PREFIX + '_results.json')}")

print("\n" + "=" * 60)
print(f"RINGKASAN {PREFIX.upper()} (conf60)")
print("=" * 60)
for name, r in all_results.items():
    print(f"  {name:<25} Acc={r['accuracy']:.4f} F1={r['macro_f1']:.4f}")


Running Late Fusion B1, B2, B3...


  B1: w=0.60 Acc=0.8019 Macro-F1=0.5133 W-F1=0.8122


  B2: w=0.50 Acc=0.8181 Macro-F1=0.5192 W-F1=0.8240


  B3: w=0.55 Acc=0.8116 Macro-F1=0.5674 W-F1=0.8208

Saved: /home/bs000716/MOTHER-TANK/TRAIN/models/frontonly_conf60/4class_tl/late_fusion_tl_results.json

RINGKASAN LATE_FUSION_TL (conf60)
  B1 Baseline               Acc=0.8019 F1=0.5133
  B2 Class Weights          Acc=0.8181 F1=0.5192
  B3 Weights+Aug            Acc=0.8116 F1=0.5674
